# ⚡ Amazon ML Challenge 2026 — Kaggle 30GB RAM Pipeline

This notebook is specifically designed for **Kaggle Notebooks (29-30 GB RAM)**.
It splits the execution into two discrete phases:
1. **Baseline Generation (CPU Mode)**: Leverages 30 GB RAM to safely build the inverted index without OOMs, extracts lexical features, and builds the baseline XGBoost.
2. **Full Pipeline (GPU Mode)**: Uses the Kaggle T4/P100 GPUs to extract heavy BGE-M3 embeddings on top of the baseline pairs, then trains the final comprehensive XGBoost model.

## 🛠️ Step 1: System Hardware Check

In [ ]:
import psutil, os
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
cpus = os.cpu_count()
print(f"KAGGLE HARDWARE: {cpus} CPU Cores | {ram_gb:.2f} GB RAM")

## 📦 Step 2: Clone Repo & Install Git LFS

In [ ]:
!apt-get install -y git-lfs > /dev/null
!git lfs install

import os
if not os.path.exists("AmazonMLChallenge2026"):
    !git clone https://github.com/priyanshu-ogdev/AmazonMLChallenge2026.git
%cd AmazonMLChallenge2026
!git pull origin main
!git lfs pull

# Fix split index parts automatically
!cat output/phase1_blocking_train/candidate_index.pkl.part* > output/phase1_blocking_train/candidate_index.pkl

## ⚙️ Step 3: Install Dependencies

In [ ]:
!pip install -q -r code/business_entity_resolution/requirements.txt
!chmod +x scripts/*.sh
!bash scripts/00_verify_environment.sh --python-path python3

## 🚀 Step 4: PHASE 1 - Train Baseline (Blocking + Lexical Features)
Run this with the notebook set to CPU or GPU. Because Kaggle gives 30GB of RAM, we explicitly allow 1 or 2 workers. The script handles this automatically now.

In [ ]:
!bash scripts/run_all_phases.sh --run-mode Full --skip-gpu --baseline-only --python-path python3

## 💾 Step 5: Save Baseline Artifacts
We copy the baseline outputs to a separate folder so they are safely backed up.

In [ ]:
!mkdir -p /kaggle/working/baseline_backup
!cp -r output/phase3_scoring_train /kaggle/working/baseline_backup/
!cp output/phase4_predictions/submission.csv /kaggle/working/baseline_backup/baseline_submission.csv
print("Baseline safely backed up to /kaggle/working/baseline_backup/")

## 🧠 Step 6: PHASE 2 - GPU Full Pipeline (BGE-M3 + Final XGBoost)
**IMPORTANT:** Make sure your Kaggle Notebook has **GPU T4x2** enabled before running this!
This step resumes the pipeline, extracts the heavy BGE-M3 neural embeddings, and trains the final XGBoost.

In [ ]:
# Run Phase 2a (BGE Dense Features) and Phase 3 (Final XGBoost Training) and Phase 4 (Decision)
!bash scripts/run_all_phases.sh --run-mode Full --from-phase 2 --python-path python3

## 📦 Step 7: Final Kaggle Output Generation

In [ ]:
import os
import shutil

zip_name = "/kaggle/working/final_submission_artifacts.zip"
!zip -r -q {zip_name} output/ -x "output/**/_tmp_worker_chunks/*" "output/**/*.tmp"
print(f"Final package saved to {zip_name}. Download it from the right-hand Kaggle output panel!")